In [1]:
tabla='ctsci10'
col_essi='fec_sol'
col_tabla='solcitafec'
essi='essi_dat_cex003_etl'

In [2]:
from decouple import config
import decouple
from sqlalchemy import create_engine
import pandas as pd
import oracledb
import sys
from sqlalchemy import text

In [15]:
#CONEXIONES
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=3", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=3"

c1= text(query)
connection2.execute(c1)

2023-05-01


In [7]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
cas_red.to_sql(name='tmp_cas_red', con=engine1, if_exists='replace', index=False)

servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios.to_sql(name='tmp_servicios', con=engine1, if_exists='replace', index=False)

areas = pd.read_sql_query(f"SELECT cod_are,des_are FROM dim_areas", con=connection2)
areas.to_sql(name='tmp_areas', con=engine1, if_exists='replace', index=False)

subacti = pd.read_sql_query(f"SELECT cod_sub,des_sub FROM dim_subacti", con=connection2)
subacti.to_sql(name='tmp_subacti', con=engine1, if_exists='replace', index=False)


activi = pd.read_sql_query(f"SELECT cod_act,des_act FROM dim_activi", con=connection2)
activi.to_sql(name='tmp_activi', con=engine1, if_exists='replace', index=False)


tipcit = pd.read_sql_query(f"SELECT cod_tci,des_tci FROM dim_tipcit", con=connection2)
tipcit.to_sql(name='tmp_tipcit', con=engine1, if_exists='replace', index=False)


cexdiapresol = pd.read_sql_query(f"SELECT cod_dps,des_dps FROM dim_cexdiapresol", con=connection2)
cexdiapresol.to_sql(name='tmp_cexdiapresol', con=engine1, if_exists='replace', index=False)

cexestreg = pd.read_sql_query(f"SELECT cod_reg,des_reg FROM dim_cexestreg", con=connection2)
cexestreg.to_sql(name='tmp_cexestreg', con=engine1, if_exists='replace', index=False)

cexestsolcit = pd.read_sql_query(f"SELECT cod_esc,des_esc FROM dim_cexestsolcit", con=connection2)
cexestsolcit.to_sql(name='tmp_cexestsolcit', con=engine1, if_exists='replace', index=False)

cexprisol = pd.read_sql_query(f"SELECT cod_pri,des_pr1 FROM dim_cexprisol", con=connection2)
cexprisol.to_sql(name='tmp_cexprisol', con=engine1, if_exists='replace', index=False)

cexturprecit = pd.read_sql_query(f"SELECT cod_tpc,des_tpc FROM dim_cexturprecit", con=connection2)
cexturprecit.to_sql(name='tmp_cexturprecit', con=engine1, if_exists='replace', index=False)

4

query_count_before = f"SELECT COUNT(*) FROM {essi}"
cant_antes = connection1.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {essi} antes de la inserción: {cant_antes}")

In [8]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [9]:
query1=f"""

INSERT INTO {essi} 
(cod_est,cod_act,cod_sub,act_med,act_med_ori,cod_are,cas_sol,cod_cas,cas_sol_ori,cod_cps,fec_cre,est_reg,fec_sol,fec_anu,fec_pre,ip_anu,ip_cre,ip_mod,mod_anu,mod_cre,fec_mod,mod_mod,num_sol,ori_sol,ori_cas,ori_sol_ori,pac_sec,cod_pri,sec_num,cod_ser,cod_tci,tip_ter,med_fis,usu_anu,usu_cre,usu_mod,cod_tpc,cod_dpc) 



SELECT 
estatensolcitacod,solcitaactcod,solcitaactespcod,solcitaactmedcitnum,solcitaactmedorinum,solcitaarehoscod,solcitacenasicitcod,solcitacenasicod,solcitacenasioricod,solcitacpscod,solcitacrefec,solcitaestregcod,solcitafec,solcitafecanu,solcitafecpref,solcitaipanucod,solcitaipcrecod,solcitaipmodcod,solcitamodanucod,solcitamodcrecod,solcitamodfec,solcitamodmodcod,solcitanum,solcitaoricenasicitcod,solcitaoricenasicod,solcitaoricenasioricod,solcitapacsecnum,solcitapricod,solcitasecnum,solcitaservhoscod,solcitatipocitacod,solcitatiptercod,solcitatramedfissolnum,solcitausuanucod,solcitausucrecod,solcitausumodcod,turprefsolcitacod,diaprefsolcitcod






FROM dblink('dbname=dl_essi user=ugaddba001ir password=U64dING23', 'SELECT * FROM {tabla} WHERE {tabla}.{col_tabla} >=''{fecha}''')

AS tmp_tbl(
	solcitanum  numeric(10,0),
	solcitafec  date,
	solcitaoricenasicod  character(1),
	solcitacenasicod  character(3),
	solcitapacsecnum  numeric(10,0),
	solcitaarehoscod  character(2),
	solcitaservhoscod  character(3),
	solcitaactcod  character(2),
	solcitaactespcod  character(3),
	solcitatipocitacod  character(1),
	solcitafecpref  date,
	diaprefsolcitcod  character(1),
	turprefsolcitacod  character(1),
	estatensolcitacod  character(1),
	solcitaoricenasioricod  character(1),
	solcitacenasioricod  character(3),
	solcitaactmedorinum  numeric(10,0),
	solcitaoricenasicitcod  character(1),
	solcitacenasicitcod  character(3),
	solcitaactmedcitnum  numeric(10,0),
	solcitaestregcod  character(1),
	solcitausucrecod  character(10),
	solcitacrefec  timestamp,
	solcitausumodcod  character(10),
	solcitamodfec  timestamp,
	solcitapricod  numeric(1,0),
	solcitaipcrecod  character(15),
	solcitausuanucod  character(10),
	solcitaipanucod  character(15),
	solcitafecanu  timestamp,
	solcitaipmodcod  character(15),
	solcitamodcrecod  character(1),
	solcitamodanucod  character(1),
	solcitamodmodcod  character(1),
	solcitasecnum  numeric(4,0),
	solcitacpscod  character(10),
	solcitatramedfissolnum  numeric(10,0),
	solcitatiptercod  character(2)

);
"""
c2= text(query1)
cursor=connection1.execute(c2)




query_count_after = f"SELECT COUNT(*) FROM {essi}"
cant_despues = connection1.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {essi} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

### Trayendo maestros

In [10]:
query1=f"""
ALTER TABLE tmp_cas_red 
ALTER COLUMN id_red TYPE CHARACTER (2),
ALTER COLUMN cod_cas TYPE CHARACTER (3),
ALTER COLUMN cod_red TYPE CHARACTER (2),
ALTER COLUMN des_red TYPE CHARACTER (60),
ALTER COLUMN des_cas TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_cas= t.des_cas,
des_red= t.des_red,
cod_red= t.cod_red
FROM tmp_cas_red t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_cas = t.cod_cas and {essi}.cod_cas IS NOT NULL and t.cod_cas IS NOT NULL ;



ALTER TABLE tmp_servicios
ALTER COLUMN cod_ser TYPE CHARACTER (3),
ALTER COLUMN des_ser TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_ser= t.des_ser

FROM tmp_servicios t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_ser = t.cod_ser and {essi}.cod_ser IS NOT NULL and t.cod_ser IS NOT NULL ;



ALTER TABLE tmp_areas
ALTER COLUMN cod_are TYPE CHARACTER (2),
ALTER COLUMN des_are TYPE CHARACTER (30);

UPDATE {essi} 
SET 
des_are= t.des_are

FROM tmp_areas t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_are = t.cod_are and {essi}.cod_are IS NOT NULL and t.cod_are IS NOT NULL ;




ALTER TABLE tmp_activi 
ALTER COLUMN cod_act TYPE CHARACTER (2),
ALTER COLUMN des_act TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_act= t.des_act

FROM tmp_activi t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_act = t.cod_act and {essi}.cod_act IS NOT NULL and t.cod_act IS NOT NULL ;


ALTER TABLE tmp_subacti
ALTER COLUMN cod_sub TYPE CHARACTER (3),
ALTER COLUMN des_sub TYPE CHARACTER (60);

UPDATE {essi} 
SET 
des_sub= t.des_sub

FROM tmp_subacti t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_sub = t.cod_sub;





















ALTER TABLE tmp_cexturprecit
ALTER COLUMN cod_tpc TYPE CHARACTER (1),
ALTER COLUMN des_tpc TYPE CHARACTER (15);

UPDATE {essi} 
SET 
des_tpc= t.des_tpc

FROM tmp_cexturprecit t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_tpc = t.cod_tpc;





ALTER TABLE tmp_cexprisol
ALTER COLUMN cod_pri TYPE CHARACTER (1),
ALTER COLUMN des_pr1 TYPE CHARACTER (5);

UPDATE {essi} 
SET 
des_pri= t.des_pr1

FROM tmp_cexprisol t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_pri = t.cod_pri;





ALTER TABLE tmp_cexestsolcit
ALTER COLUMN cod_esc TYPE CHARACTER (1),
ALTER COLUMN des_esc TYPE CHARACTER (25);

UPDATE {essi} 
SET 
des_est= t.des_esc

FROM tmp_cexestsolcit t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_est = t.cod_esc;






ALTER TABLE tmp_tipcit
ALTER COLUMN cod_tci TYPE CHARACTER (1),
ALTER COLUMN des_tci TYPE CHARACTER (15);

UPDATE {essi} 
SET 
des_tci= t.des_tci

FROM tmp_tipcit t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_tci = t.cod_tci;



ALTER TABLE tmp_cexdiapresol
ALTER COLUMN cod_dps TYPE CHARACTER (1),
ALTER COLUMN des_dps TYPE CHARACTER (15);

UPDATE {essi} 
SET 
des_dpc= t.des_dps

FROM tmp_cexdiapresol t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_dpc = t.cod_dps;






ALTER TABLE tmp_cexestreg
ALTER COLUMN cod_reg TYPE CHARACTER (1),
ALTER COLUMN des_reg TYPE CHARACTER (30);

UPDATE {essi} 
SET 
des_reg= t.des_reg

FROM tmp_cexestreg t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.est_reg = t.cod_reg;



ALTER TABLE tmp_cas_red 
ALTER COLUMN id_red TYPE CHARACTER (2),
ALTER COLUMN cod_cas TYPE CHARACTER (3),
ALTER COLUMN cod_red TYPE CHARACTER (2),
ALTER COLUMN des_red TYPE CHARACTER (60),
ALTER COLUMN des_cas TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_cso= t.des_cas
FROM tmp_cas_red t
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cas_sol = t.cod_cas and {essi}.cas_sol IS NOT NULL and t.cod_cas IS NOT NULL ;



DROP TABLE tmp_areas;
DROP TABLE tmp_servicios;
DROP TABLE tmp_cas_red;
DROP TABLE tmp_activi;
DROP TABLE tmp_subacti;
DROP TABLE tmp_cexturprecit;
DROP TABLE tmp_cexestsolcit;
DROP TABLE tmp_cexprisol;
DROP TABLE tmp_cexdiapresol;
DROP TABLE tmp_cexestreg;
DROP TABLE tmp_tipcit;


"""
c2= text(query1)
cursor=connection1.execute(c2)

In [11]:
connection1.close()
connection2.close()
connection3.close()